In [ ]:
using Pkg
Pkg.activate("/Users/bursche/Documents/GitHub/JPEC_BCRIT")
Base.active_project()

using GeneralizedPerturbedEquilibrium
using GeneralizedPerturbedEquilibrium: Analysis
using Printf

using Plots
default(
    fontfamily="Georgia",
    margin=12Plots.mm,
    size=(800, 500),
    dpi=150
)

In [ ]:
h5path = "/Users/bursche/Documents/GitHub/JPEC_BCRIT/examples/DIIID-like_SLAYER_example/gpec.h5"

In [ ]:
using HDF5

h5open(h5path, "r") do file
    println("Top-level keys:")
    println(collect(keys(file)))

    if haskey(file, "Tearing")
        println("\nKeys in Tearing:")
        println(collect(keys(file["Tearing"])))

        if haskey(file["Tearing"], "CriticalResonantField")
            println("\nKeys in Tearing/CriticalResonantField:")
            println(collect(keys(file["Tearing"]["CriticalResonantField"])))
        end
    else
        println("No Tearing group found.")
    end
end

In [ ]:
# Print the contents of the Tearing/CriticalResonantField group
h5open(h5path, "r") do file
    if haskey(file, "Tearing") && haskey(file["Tearing"], "CriticalResonantField")
        println("\nContents of Tearing/CriticalResonantField:")
        # Iterate over the keys in the Tearing/CriticalResonantField group and print their names and types
        for key in keys(file["Tearing"]["CriticalResonantField"])
            dataset = file["Tearing"]["CriticalResonantField"][key]
            println("Key: $key, Type: $(typeof(dataset))")
            # print data 
            if isa(dataset, HDF5.Dataset)
                data = read(dataset)
                println("Data: $data")
            end
        end
    else
        println("No Tearing/CriticalResonantField group found.")
    end
    # Print bcrit data as .2e (in one line)
    println("\nBcrit data: $(join([@sprintf("%.2e", x) for x in read(file["Tearing"]["CriticalResonantField"]["br_crit"])], " "))")
end



In [ ]:
println(keys(h5open(h5path, "r")["Tearing"]["CriticalResonantField"]["Scan"]["surface_1"]))

In [ ]:
function bcrit_diag_plots(Δs, Qs, bal,name)

    p1 = plot(Qs, imag.(Δs), label="Im(Δ)", lw=2)
    #plot!(p1, Qs, real.(Δs), label="Re(Δ)", lw=2)
    xlabel!(p1, "Q")
    ylabel!(p1, "Δ")
    title!(p1, "Inner-layer Δ(Q) - $name")

    p2 = plot(Qs, real.(bal), label="Re(balance)", lw=2)
    plot!(p2, Qs, imag.(bal), label="Im(balance)", lw=2)
    xlabel!(p2, "Q")
    ylabel!(p2, "balance")
    title!(p2, "2P(Q0-Q)/jxb - $name")

    plot(p1, p2, layout=(2,1), size=(800, 1000))


end

In [ ]:
# Plot bcrit vs q surface

# print all contents of bcrit /scan /surface 1/
h5open(h5path, "r") do file
    bcrit = file["Tearing"]["CriticalResonantField"]

    for i in 1:6
        ss = "surface_$i"

        delta  = read(bcrit["Scan"][ss]["delta"])
        Q      = read(bcrit["Scan"][ss]["Q"])
        balance = read(bcrit["Scan"][ss]["balance"])

        display(bcrit_diag_plots(delta, Q, balance, "surface_$i"))
    end
end

In [ ]:
p_eq = Analysis.Equilibrium.plot_equilibrium_summary(h5path)
p_ffs = Analysis.ForceFreeStates.plot_ffs_summary(h5path)
#p_pe = Analysis.PerturbedEquilibrium.plot_perturbed_equilibrium_summary(h5path)

display(p_eq)
display(p_ffs)
#display(p_pe)